# Legislative Content Extraction (ADK) - Evaluation Smoke Test

Fetch the dataset from Langfuse and run the agent on the first item to verify the pipeline works end-to-end.

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

# Ensure cwd is the repo root so relative paths work
REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
# Walk up until we find pyproject.toml
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

load_dotenv(verbose=True)
print(f"Working directory: {Path.cwd()}")

## 1. Fetch dataset from Langfuse

In [ ]:
from aieng.agent_evals.async_client_manager import AsyncClientManager

client_manager = AsyncClientManager.get_instance()
langfuse_client = client_manager.langfuse_client

DATASET_NAME = "legislative-docs-ID"
FILES_DIR = Path("implementations/legislative_content_extraction/files").resolve()
MAX_CONCURRENCY = 3

dataset = langfuse_client.get_dataset(DATASET_NAME)
print(f"Dataset name: {dataset.name}")
print(f"Items:   {len(dataset.items)}")
print(f"Files dir:     {FILES_DIR}")
print(f"Concurrency:   {MAX_CONCURRENCY}")

## 2. Inspect the first item

In [ ]:
first_item = dataset.items[0]

print("Input:")
print(json.dumps(first_item.input, indent=2))
print("\nExpected output:")
print(json.dumps(first_item.expected_output, indent=2))
print("\nMetadata:")
print(json.dumps(first_item.metadata, indent=2))

## 3. Run evaluation on the dataset

In [ ]:
from aieng.agent_evals.legislative_content_extraction.evaluation.offline import evaluate
from datetime import datetime, timezone
iso_timestamp = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

# add name to identify your run
custom_run_name = "eval"

# add description to identify your run
custom_description = "evals for ID"

await evaluate(
    dataset_name=DATASET_NAME,
    files_dir=FILES_DIR,
    max_concurrency=MAX_CONCURRENCY,
    run_name=f"{custom_run_name} - {iso_timestamp}",
    description=custom_description
)